<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/exercice_keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation des bibliothèques
Installez (ou mettez à jour) les bibliothèques nécessaires pour exécuter le notebook : `keras` et `keras-hub`.

In [ ]:
!pip install keras keras-hub --upgrade -q

### Configuration du backend Keras
Configurez Keras pour utiliser le backend **TENSORFLOW** en définissant la variable d’environnement `KERAS_BACKEND` avant d’importer Keras.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

#### Modèle Sequential (définition directe)
Construisez un modèle Keras `Sequential` composé de :
- une couche `Dense(64, activation="relu")`,
- suivie d’une couche `Dense(10, activation="softmax")`.

#### Modèle Sequential (construction progressive)
Recréez le même modèle `Sequential`, mais cette fois en :
1. instanciant `keras.Sequential()` sans couches,
2. ajoutant les couches une par une avec `model.add(...)`.

In [ ]:
model = keras.Sequential()
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(10, activation="softmax"))

#### Inspection des poids du modèle
Affichez `model.weights` :
1. avant d’avoir “build” le modèle,
2. puis après avoir appelé `model.build(input_shape=(None, 3))`.
Expliquez brièvement la différence observée.

#### Résumé du modèle
Affichez l’architecture du modèle avec `model.summary()` (avec une largeur de ligne adaptée).

In [ ]:
model.summary(line_length=80)

#### Nommage du modèle et des couches
Créez un modèle `Sequential` en donnant :
- un nom au modèle (ex: `"my_example_model"`),
- un nom explicite à la première couche dense,
- un nom explicite à la dernière couche dense,
puis construisez le modèle pour une entrée de dimension 3 et affichez `summary()`.

In [ ]:
model = keras.Sequential(name="my_example_model")
model.add(layers.Dense(64, activation="relu", name="my_first_layer"))
model.add(layers.Dense(10, activation="softmax", name="my_last_layer"))
model.build((None, 3))
model.summary(line_length=80)

#### Ajout d’une couche d’entrée dans Sequential
Construisez un modèle `Sequential` qui commence par une entrée `keras.Input(shape=(3,))`,
puis ajoute une couche `Dense(64, activation="relu")`.
Affichez le résumé du modèle.

#### Compléter un modèle Sequential après sa création
Ajoutez ensuite une couche `Dense(10, activation="softmax")` au modèle précédent,
puis réaffichez `model.summary()` pour constater la mise à jour.

##### Functional API : exemple simple
Construisez un modèle Keras avec l’API fonctionnelle :
- une entrée `keras.Input(shape=(3,), name="my_input")`,
- une couche Dense(64, relu) appliquée à l’entrée,
- une couche Dense(10, softmax) appliquée aux features,
- puis créez un objet `keras.Model` (avec un nom explicite).

In [ ]:
inputs = keras.Input(shape=(3,), name="my_input")
features = layers.Dense(64, activation="relu")(inputs)
outputs = layers.Dense(10, activation="softmax")(features)
model = keras.Model(inputs=inputs, outputs=outputs, name="my_functional_model")

##### Inspecter le tenseur d’entrée
Créez un tenseur d’entrée avec `keras.Input(...)` puis affichez :
- sa `shape`,
- son `dtype`.

##### Vérifier les dimensions intermédiaires
Appliquez une couche `Dense(64, relu)` à l’entrée, stockez le résultat dans `features`,
puis affichez `features.shape`.

##### Création et résumé du modèle fonctionnel
Créez la sortie `Dense(10, softmax)` à partir de `features`,
instanciez `keras.Model(inputs=..., outputs=...)`,
puis affichez `model.summary()`.

##### Modèle multi-entrées / multi-sorties
Construisez un modèle fonctionnel qui prend trois entrées :
- `title` : vecteur de taille `vocabulary_size`,
- `text_body` : vecteur de taille `vocabulary_size`,
- `tags` : vecteur de taille `num_tags`.

Concaténez ces entrées, appliquez une couche Dense(64, relu),
puis produisez deux sorties :
- `priority` : Dense(1, sigmoid),
- `department` : Dense(num_departments, softmax).
Créez ensuite le `keras.Model` correspondant.

In [ ]:
vocabulary_size = 10000
num_tags = 100
num_departments = 4

title = keras.Input(shape=(vocabulary_size,), name="title")
text_body = keras.Input(shape=(vocabulary_size,), name="text_body")
tags = keras.Input(shape=(num_tags,), name="tags")

features = layers.Concatenate()([title, text_body, tags])
features = layers.Dense(64, activation="relu", name="dense_features")(features)

priority = layers.Dense(1, activation="sigmoid", name="priority")(features)
department = layers.Dense(
    num_departments, activation="softmax", name="department"
)(features)

model = keras.Model(
    inputs=[title, text_body, tags],
    outputs=[priority, department],
)

##### Entraînement (format listes)
Générez des données aléatoires (NumPy) cohérentes avec le modèle multi-entrées / multi-sorties :
- entrées binaires pour `title`, `text_body`, `tags`,
- une cible réelle pour `priority`,
- une cible entière pour `department`.

Compilez le modèle avec :
- optimizer Adam,
- deux fonctions de perte (MSE pour `priority`, sparse categorical crossentropy pour `department`),
- des métriques adaptées (MAE pour `priority`, accuracy pour `department`).

Lancez `fit` sur 1 epoch, puis `evaluate`, puis `predict`.

In [ ]:
import numpy as np

num_samples = 1280

title_data = np.random.randint(0, 2, size=(num_samples, vocabulary_size))
text_body_data = np.random.randint(0, 2, size=(num_samples, vocabulary_size))
tags_data = np.random.randint(0, 2, size=(num_samples, num_tags))

priority_data = np.random.random(size=(num_samples, 1))
department_data = np.random.randint(0, num_departments, size=(num_samples, 1))

model.compile(
    optimizer="adam",
    loss=["mean_squared_error", "sparse_categorical_crossentropy"],
    metrics=[["mean_absolute_error"], ["accuracy"]],
)
model.fit(
    [title_data, text_body_data, tags_data],
    [priority_data, department_data],
    epochs=1,
)
model.evaluate(
    [title_data, text_body_data, tags_data], [priority_data, department_data]
)
priority_preds, department_preds = model.predict(
    [title_data, text_body_data, tags_data]
)

##### Entraînement (format dictionnaires)
Reprenez exactement le même entraînement, mais cette fois :
- passez les entrées à `fit/evaluate/predict` sous forme de dictionnaire `{nom: données}`,
- définissez `loss` et `metrics` sous forme de dictionnaires indexés par les noms des sorties.

In [ ]:
model.compile(
    optimizer="adam",
    loss={
        "priority": "mean_squared_error",
        "department": "sparse_categorical_crossentropy",
    },
    metrics={
        "priority": ["mean_absolute_error"],
        "department": ["accuracy"],
    },
)
model.fit(
    {"title": title_data, "text_body": text_body_data, "tags": tags_data},
    {"priority": priority_data, "department": department_data},
    epochs=1,
)
model.evaluate(
    {"title": title_data, "text_body": text_body_data, "tags": tags_data},
    {"priority": priority_data, "department": department_data},
)
priority_preds, department_preds = model.predict(
    {"title": title_data, "text_body": text_body_data, "tags": tags_data}
)

###### Visualiser le graphe du modèle
Utilisez `keras.utils.plot_model` pour exporter un schéma du modèle dans un fichier image.
Puis faites une seconde exportation en affichant :
- les formes des tenseurs (`show_shapes=True`),
- les noms des couches (`show_layer_names=True`).

###### Explorer les couches d’un modèle fonctionnel
Affichez la liste `model.layers`, puis affichez :
- l’entrée (`input`) d’une couche donnée,
- la sortie (`output`) de cette même couche.

In [ ]:
model.layers[3].input

###### Ajouter une nouvelle sortie à partir d’une feature intermédiaire
Récupérez la sortie d’une couche intermédiaire du modèle (ex: `model.layers[...] .output`),
ajoutez une nouvelle tête de classification `difficulty = Dense(3, softmax, name="difficulty")`,
puis construisez un **nouveau modèle** ayant les mêmes entrées mais **trois sorties**
(priority, department, difficulty). Exportez ensuite son schéma avec `plot_model`.

In [ ]:
features = model.layers[4].output
difficulty = layers.Dense(3, activation="softmax", name="difficulty")(features)

new_model = keras.Model(
    inputs=[title, text_body, tags], outputs=[priority, department, difficulty]
)

In [ ]:
keras.utils.plot_model(
    new_model,
    "updated_ticket_classifier.png",
    show_shapes=True,
    show_layer_names=True,
)

##### Subclassing : créer une classe de modèle
Créez une classe `CustomerTicketModel(keras.Model)` qui :
- définit dans `__init__` les couches nécessaires (Concatenate, Dense(64, relu), Dense(1,sigmoid), Dense(num_departments,softmax)),
- implémente `call(self, inputs)` en récupérant `inputs["title"]`, `inputs["text_body"]`, `inputs["tags"]`,
- concatène, calcule les features, puis retourne `(priority, department)`.

In [ ]:
class CustomerTicketModel(keras.Model):
    def __init__(self, num_departments):
        super().__init__()
        self.concat_layer = layers.Concatenate()
        self.mixing_layer = layers.Dense(64, activation="relu")
        self.priority_scorer = layers.Dense(1, activation="sigmoid")
        self.department_classifier = layers.Dense(
            num_departments, activation="softmax"
        )

    def call(self, inputs):
        title = inputs["title"]
        text_body = inputs["text_body"]
        tags = inputs["tags"]

        features = self.concat_layer([title, text_body, tags])
        features = self.mixing_layer(features)
        priority = self.priority_scorer(features)
        department = self.department_classifier(features)
        return priority, department

##### Tester un modèle subclassé
Instanciez `CustomerTicketModel(num_departments=4)` puis appelez-le sur un dictionnaire d’entrées
pour obtenir `priority` et `department`.

In [ ]:
model = CustomerTicketModel(num_departments=4)

priority, department = model(
    {"title": title_data, "text_body": text_body_data, "tags": tags_data}
)

##### Entraîner un modèle subclassé
Compilez le modèle subclassé avec Adam, les deux pertes (MSE + sparse categorical crossentropy)
et les métriques associées, puis lancez `fit`, `evaluate` et `predict` sur les données synthétiques.

In [ ]:
model.compile(
    optimizer="adam",
    loss=["mean_squared_error", "sparse_categorical_crossentropy"],
    metrics=[["mean_absolute_error"], ["accuracy"]],
)
model.fit(
    {"title": title_data, "text_body": text_body_data, "tags": tags_data},
    [priority_data, department_data],
    epochs=1,
)
model.evaluate(
    {"title": title_data, "text_body": text_body_data, "tags": tags_data},
    [priority_data, department_data],
)
priority_preds, department_preds = model.predict(
    {"title": title_data, "text_body": text_body_data, "tags": tags_data}
)

#### Composant réutilisable : Classifier
Créez une classe `Classifier(keras.Model)` qui :
- choisit `Dense(1, sigmoid)` si `num_classes==2`,
- sinon `Dense(num_classes, softmax)`,
- et renvoie `self.dense(inputs)` dans `call`.

Utilisez ensuite cette classe dans un modèle fonctionnel :
Input(3) -> Dense(64, relu) -> Classifier(num_classes=10).

#### Modèle dans un modèle
Créez d’abord un petit modèle fonctionnel `binary_classifier` :
Input(64) -> Dense(1, sigmoid).

Puis créez une classe `MyModel(keras.Model)` qui :
- contient une couche Dense(64, relu),
- contient `binary_classifier` comme attribut,
- et dans `call` applique Dense puis le sous-modèle.

In [ ]:
inputs = keras.Input(shape=(64,))
outputs = layers.Dense(1, activation="sigmoid")(inputs)
binary_classifier = keras.Model(inputs=inputs, outputs=outputs)

class MyModel(keras.Model):
    def __init__(self, num_classes=2):
        super().__init__()
        self.dense = layers.Dense(64, activation="relu")
        self.classifier = binary_classifier

    def call(self, inputs):
        features = self.dense(inputs)
        return self.classifier(features)

model = MyModel()

### Entraînement standard avec fit()
Chargez MNIST, aplatissez et normalisez les images (float32 / 255),
faites une séparation train/validation,
définissez une fonction `get_mnist_model()` qui construit :
Input(784) -> Dense(512, relu) -> Dropout(0.5) -> Dense(10, softmax).

Compilez avec Adam + sparse categorical crossentropy + accuracy,
entraîne avec `fit` (3 epochs, validation_data),
puis évaluez sur test et générez des prédictions.

In [ ]:
from keras.datasets import mnist

def get_mnist_model():
    inputs = keras.Input(shape=(28 * 28,))
    features = layers.Dense(512, activation="relu")(inputs)
    features = layers.Dropout(0.5)(features)
    outputs = layers.Dense(10, activation="softmax")(features)
    model = keras.Model(inputs, outputs)
    return model

(images, labels), (test_images, test_labels) = mnist.load_data()
images = images.reshape((60000, 28 * 28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255
train_images, val_images = images[10000:], images[:10000]
train_labels, val_labels = labels[10000:], labels[:10000]

model = get_mnist_model()
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    train_images,
    train_labels,
    epochs=3,
    validation_data=(val_images, val_labels),
)
test_metrics = model.evaluate(test_images, test_labels)
predictions = model.predict(test_images)

#### Métrique personnalisée
Créez une classe `RootMeanSquaredError` qui hérite de `keras.metrics.Metric` et qui :
- crée deux variables d’état (somme des MSE et nombre d’échantillons),
- implémente `update_state`, `result`, et `reset_state`,
- calcule une RMSE à partir de y_true et y_pred.

In [ ]:
from keras import ops

class RootMeanSquaredError(keras.metrics.Metric):
    def __init__(self, name="rmse", **kwargs):
        super().__init__(name=name, **kwargs)
        self.mse_sum = self.add_weight(name="mse_sum", initializer="zeros")
        self.total_samples = self.add_weight(
            name="total_samples", initializer="zeros"
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = ops.one_hot(y_true, num_classes=ops.shape(y_pred)[1])
        mse = ops.sum(ops.square(y_true - y_pred))
        self.mse_sum.assign_add(mse)
        num_samples = ops.shape(y_pred)[0]
        self.total_samples.assign_add(num_samples)

    def result(self):
        return ops.sqrt(self.mse_sum / self.total_samples)

    def reset_state(self):
        self.mse_sum.assign(0.)
        self.total_samples.assign(0.)

#### Utiliser une métrique custom dans un entraînement
Reprenez le modèle MNIST, compilez-le en ajoutant `RootMeanSquaredError()` à la liste des métriques,
puis entraînez et évaluez le modèle.

Dans le contexte des frameworks de Machine Learning comme Keras, les callbacks sont des objets spéciaux qui peuvent être appliqués pendant l'entraînement d'un modèle pour exécuter des actions à différentes étapes du processus (début/fin d'epoch, début/fin de batch, etc.). Ils permettent d'automatiser et de personnaliser le comportement de l'entraînement sans modifier le code source du modèle ou de l'algorithme d'entraînement.

**Exemples d'utilisation courante des callbacks Keras :**

*   **`ModelCheckpoint`**: Sauvegarde le modèle à intervalles réguliers ou lorsque certaines métriques s'améliorent.
*   **`EarlyStopping`**: Arrête l'entraînement si la performance sur un ensemble de validation ne s'améliore plus pendant un certain nombre d'epochs (`patience`).
*   **`TensorBoard`**: Permet de visualiser les courbes d'apprentissage, les métriques, l'architecture du modèle, etc., dans l'interface TensorBoard.
*   **`ReduceLROnPlateau`**: Réduit le taux d'apprentissage lorsque le modèle stagne sur la métrique de validation.
*   **`CSVLogger`**: Enregistre les métriques d'entraînement et de validation dans un fichier CSV.
*   **Callbacks personnalisés**: Vous pouvez créer vos propres callbacks pour des besoins spécifiques (par exemple, pour visualiser des activations, envoyer des notifications, ou effectuer des calculs complexes à chaque epoch).

**Pourquoi utiliser des callbacks ?**

*   **Automatisation**: Ils automatisent des tâches répétitives pendant l'entraînement.
*   **Flexibilité**: Ils permettent d'ajouter des fonctionnalités sans modifier la boucle d'entraînement principale.
*   **Surveillance et Debugging**: Ils facilitent le suivi des performances et l'identification des problèmes.
*   **Optimisation**: Ils aident à optimiser le processus d'entraînement (ex: `EarlyStopping`, `ReduceLROnPlateau`).


##### Callbacks : EarlyStopping et ModelCheckpoint
Créez une liste de callbacks contenant :
- `EarlyStopping(monitor="accuracy", patience=1)`,
- `ModelCheckpoint(filepath="checkpoint_path.keras", monitor="val_loss", save_best_only=True)`.

Entraînez le modèle MNIST pendant 10 epochs avec ces callbacks et une validation.

In [ ]:
callbacks_list = [
    keras.callbacks.EarlyStopping(
        monitor="accuracy",
        patience=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="checkpoint_path.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
]
model = get_mnist_model()
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    train_images,
    train_labels,
    epochs=10,
    callbacks=callbacks_list,
    validation_data=(val_images, val_labels),
)

##### Rechargement d’un modèle sauvegardé
Rechargez le modèle sauvegardé par `ModelCheckpoint` avec `keras.models.load_model(...)`.

In [ ]:
model = keras.models.load_model("checkpoint_path.keras")

#### Callback personnalisé
Créez une callback `LossHistory(keras.callbacks.Callback)` qui :
- stocke les pertes batch par batch pendant une epoch,
- à la fin de chaque epoch, trace la courbe (matplotlib) et sauvegarde une image `plot_at_epoch_{epoch}`.

In [ ]:
from matplotlib import pyplot as plt

class LossHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs):
        self.per_batch_losses = []

    def on_batch_end(self, batch, logs):
        self.per_batch_losses.append(logs.get("loss"))

    def on_epoch_end(self, epoch, logs):
        plt.clf()
        plt.plot(
            range(len(self.per_batch_losses)),
            self.per_batch_losses,
            label="Training loss for each batch",
        )
        plt.xlabel(f"Batch (epoch {epoch})")
        plt.ylabel("Loss")
        plt.legend()
        plt.savefig(f"plot_at_epoch_{epoch}", dpi=300)
        self.per_batch_losses = []

#### Entraîner avec une callback custom
Entraînez le modèle MNIST sur 10 epochs en passant `callbacks=[LossHistory()]`
et en utilisant des données de validation.

#### TensorBoard
Ajoutez un callback `TensorBoard(log_dir="...")` à l’entraînement du modèle MNIST
et entraînez pendant 10 epochs avec validation.

In [ ]:
model = get_mnist_model()
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

tensorboard = keras.callbacks.TensorBoard(
    log_dir="/full_path_to_your_log_dir",
)
model.fit(
    train_images,
    train_labels,
    epochs=10,
    validation_data=(val_images, val_labels),
    callbacks=[tensorboard],
)

#### Affichage TensorBoard dans le notebook
Chargez l’extension TensorBoard puis lancez l’affichage en pointant vers le même `log_dir`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /full_path_to_your_log_dir

##### Train step TensorFlow
Dans une cellule exécutée uniquement si le backend est TensorFlow :
- importez TensorFlow,
- instanciez le modèle MNIST, la loss `SparseCategoricalCrossentropy` et l’optimizer Adam,
- écrivez une fonction `train_step(inputs, targets)` qui :
  - calcule les prédictions en mode entraînement,
  - calcule la loss,
  - calcule les gradients via `tf.GradientTape`,
  - applique les gradients à `model.trainable_weights`,
  - renvoie la loss.

Lorsque l’on utilise `model.fit()`, Keras gère automatiquement :

- le calcul des prédictions,
- le calcul de la fonction de perte,
- le calcul des gradients,
- la mise à jour des poids avec l’optimiseur.

Autrement dit, la rétropropagation est faite en interne.  
On ne voit pas `GradientTape`, mais il est utilisé dans les coulisses.

En revanche, lorsque l’on écrit une boucle d’entraînement personnalisée (custom training loop),  
on doit gérer nous-même :

1. le passage avant (forward pass),
2. le calcul de la perte,
3. le calcul des gradients,
4. la mise à jour des poids.

`tf.GradientTape` est l’outil qui permet de calculer les gradients à l’étape 3.

On l’utilise donc uniquement quand on veut contrôler manuellement le processus d’entraînement, par exemple pour :

- modifier le calcul de la perte,
- implémenter un entraînement particulier,
- expérimenter avec des stratégies avancées.

In [ ]:
%%backend tensorflow
import tensorflow as tf

model = get_mnist_model()
loss_fn = keras.losses.SparseCategoricalCrossentropy()
optimizer = keras.optimizers.Adam()

def train_step(inputs, targets):
    with tf.GradientTape() as tape:
        predictions = model(inputs, training=True)
        loss = loss_fn(targets, predictions)
    gradients = tape.gradient(loss, model.trainable_weights)
    optimizer.apply(gradients, model.trainable_weights)
    return loss

##### Test du train_step TensorFlow
Créez un batch de taille 32 à partir de `train_images` et `train_labels`,
appelez `train_step` et stockez la loss obtenue.

##### Train step PyTorch
Dans une cellule exécutée uniquement si le backend est PyTorch :
- importez torch,
- instanciez modèle, loss et optimizer,
- écrivez `train_step(inputs, targets)` qui :
  - calcule prédictions et loss,
  - fait `loss.backward()`,
  - récupère les gradients des poids du modèle,
  - applique l’optimizer (dans un bloc `torch.no_grad()`),
  - remet les gradients à zéro,
  - renvoie la loss.

In [ ]:
%%backend torch
import torch

model = get_mnist_model()
loss_fn = keras.losses.SparseCategoricalCrossentropy()
optimizer = keras.optimizers.Adam()

def train_step(inputs, targets):
    predictions = model(inputs, training=True)
    loss = loss_fn(targets, predictions)
    loss.backward()
    gradients = [weight.value.grad for weight in model.trainable_weights]
    with torch.no_grad():
        optimizer.apply(gradients, model.trainable_weights)
    model.zero_grad()
    return loss

##### Test du train_step PyTorch
Comme pour TensorFlow : créez un batch de 32, appelez `train_step`, récupérez la loss.

#### Utilisation manuelle d’une métrique
Instanciez `keras.metrics.SparseCategoricalAccuracy`,
définissez `targets` et `predictions` (tableau one-hot),
appelez `update_state`, puis affichez `metric.result()`.

In [ ]:
from keras import ops # framework multi-backend

metric = keras.metrics.SparseCategoricalAccuracy()
targets = ops.array([0, 1, 2])
predictions = ops.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
metric.update_state(targets, predictions)
current_result = metric.result()
print(f"result: {current_result:.2f}")

#### Suivre une moyenne avec keras.metrics.Mean
Créez un tableau de valeurs, mettez à jour une métrique `Mean()` dans une boucle,
puis affichez la moyenne finale.

In [ ]:
values = ops.array([0, 1, 2, 3, 4])
mean_tracker = keras.metrics.Mean()
for value in values:
    mean_tracker.update_state(value)
print(f"Mean of values: {mean_tracker.result():.2f}")

#### Métriques en mode stateless
Montrez comment utiliser une métrique en mode stateless :
- récupérez `metric.variables`,
- mettez à jour via `metric.stateless_update_state(...)`,
- récupérez le résultat via `metric.stateless_result(...)`,
- puis réinitialisez via `metric.stateless_reset_state()`.

In [ ]:
metric = keras.metrics.SparseCategoricalAccuracy()
targets = ops.array([0, 1, 2])
predictions = ops.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])

metric_variables = metric.variables
metric_variables = metric.stateless_update_state(
    metric_variables, targets, predictions
)
current_result = metric.stateless_result(metric_variables)
print(f"result: {current_result:.2f}")

metric_variables = metric.stateless_reset_state()

##### Customiser fit() (TensorFlow)
Dans une cellule TensorFlow :
- définissez une loss globale + un tracker `keras.metrics.Mean(name="loss")`,
- créez une classe `CustomModel(keras.Model)` qui surcharge `train_step(self, data)` :
  - calcule predictions et loss,
  - calcule/applique gradients,
  - met à jour `loss_tracker`,
  - renvoie un dictionnaire de logs.
- redéfinissez la propriété `metrics` pour retourner `[loss_tracker]`.

In [ ]:
%%backend tensorflow
import keras
from keras import layers

loss_fn = keras.losses.SparseCategoricalCrossentropy()
loss_tracker = keras.metrics.Mean(name="loss")

class CustomModel(keras.Model):
    def train_step(self, data):
        inputs, targets = data
        with tf.GradientTape() as tape:
            predictions = self(inputs, training=True)
            loss = loss_fn(targets, predictions)
        gradients = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply(gradients, self.trainable_weights)

        loss_tracker.update_state(loss)
        return {"loss": loss_tracker.result()}

    @property
    def metrics(self):
        return [loss_tracker]

##### Entraîner le CustomModel (TensorFlow)
Écrivez `get_custom_model()` qui construit le même réseau MNIST
mais instancie `CustomModel(inputs, outputs)` puis compile avec Adam.
Entraînez ensuite avec `fit(..., epochs=3)`.

##### Customiser fit() (PyTorch)
Dans une cellule PyTorch :
- créez la classe `CustomModel(keras.Model)` qui surcharge `train_step` en utilisant `loss.backward()`,
- récupérez les gradients des poids,
- appliquez l’optimizer dans `torch.no_grad()`,
- mettez à jour `loss_tracker`,
- et exposez `metrics` -> `[loss_tracker]`.

In [ ]:
%%backend torch
import keras
from keras import layers

loss_fn = keras.losses.SparseCategoricalCrossentropy()
loss_tracker = keras.metrics.Mean(name="loss")

class CustomModel(keras.Model):
    def train_step(self, data):
        inputs, targets = data
        predictions = self(inputs, training=True)
        loss = loss_fn(targets, predictions)

        loss.backward()
        trainable_weights = [v for v in self.trainable_weights]
        gradients = [v.value.grad for v in trainable_weights]

        with torch.no_grad():
            self.optimizer.apply(gradients, trainable_weights)

        loss_tracker.update_state(loss)
        return {"loss": loss_tracker.result()}

    @property
    def metrics(self):
        return [loss_tracker]

##### Entraîner le CustomModel (PyTorch)
Construisez `get_custom_model()` (même architecture MNIST),
instanciez `CustomModel`, compilez avec Adam, puis entraînez 3 epochs.

##### train_step avec métriques (TensorFlow)
Redéfinissez `train_step` pour :
- calculer `loss = self.compute_loss(...)`,
- appliquer les gradients,
- parcourir `self.metrics` :
  - si `metric.name == "loss"` -> update_state(loss),
  - sinon -> update_state(targets, predictions),
- renvoyer un dictionnaire `{m.name: m.result()}`.

In [ ]:
%%backend tensorflow
import keras
from keras import layers

class CustomModel(keras.Model):
    def train_step(self, data):
        inputs, targets = data
        with tf.GradientTape() as tape:
            predictions = self(inputs, training=True)
            loss = self.compute_loss(y=targets, y_pred=predictions)

        gradients = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply(gradients, self.trainable_weights)

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(loss)
            else:
                metric.update_state(targets, predictions)

        return {m.name: m.result() for m in self.metrics}

In [ ]:
%%backend tensorflow
def get_custom_model():
    inputs = keras.Input(shape=(28 * 28,))
    features = layers.Dense(512, activation="relu")(inputs)
    features = layers.Dropout(0.5)(features)
    outputs = layers.Dense(10, activation="softmax")(features)
    model = CustomModel(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy()],
    )
    return model

model = get_custom_model()
model.fit(train_images, train_labels, epochs=3)

##### train_step avec métriques (PyTorch)
Reprenez le même schéma que TensorFlow, mais avec `loss.backward()` et application des gradients
à la manière PyTorch, puis mise à jour des métriques dans une boucle.

In [ ]:
%%backend torch
import keras
from keras import layers

class CustomModel(keras.Model):
    def train_step(self, data):
        inputs, targets = data
        predictions = self(inputs, training=True)
        loss = self.compute_loss(y=targets, y_pred=predictions)

        loss.backward()
        trainable_weights = [v for v in self.trainable_weights]
        gradients = [v.value.grad for v in trainable_weights]

        with torch.no_grad():
            self.optimizer.apply(gradients, trainable_weights)

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(loss)
            else:
                metric.update_state(targets, predictions)

        return {m.name: m.result() for m in self.metrics}

In [ ]:
%%backend torch
def get_custom_model():
    inputs = keras.Input(shape=(28 * 28,))
    features = layers.Dense(512, activation="relu")(inputs)
    features = layers.Dropout(0.5)(features)
    outputs = layers.Dense(10, activation="softmax")(features)
    model = CustomModel(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy()],
    )
    return model

model = get_custom_model()
model.fit(train_images, train_labels, epochs=3)